# DS_Day01_37: Digital Marketing — Which Campaign Works?
### End-to-End Performance Evaluation, EDA, Decision Tree Modeling & Budget Optimization

**Student Tasks Addressed:**
1. Clean and validate dataset
2. Descriptive statistics & exploratory data analysis
3. Compute CTR, Conversion Rate, CPA, and ROAS
4. Compare campaign performance across metrics and customer segments
5. Investigate whether high click-through rate translates to high revenue
6. Identify high-cost, low-impact campaigns
7. Train and evaluate a Decision Tree model
8. Formulate 5 to 7 key insights and a practical action plan

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)

## 1. Load and Validate Raw Data

In [ ]:
raw_path = os.path.join("..", "data", "raw_campaign_data.csv")
df_raw = pd.read_csv(raw_path)
print(f"Raw dataset shape: {df_raw.shape}")
df_raw.head()

## 2. Data Cleaning & KPI Calculations
- $\text{CTR (\%)} = (\text{Clicks} / \text{Impressions}) \times 100$
- $\text{Conversion Rate (\%)} = (\text{Conversions} / \text{Clicks}) \times 100$
- $\text{CPA} = \text{Campaign\_Cost} / \text{Conversions}$
- $\text{ROAS} = \text{Revenue} / \text{Campaign\_Cost}$

In [ ]:
# Drop anomalies & missing rows
df = df_raw.dropna(subset=["Campaign_Cost", "Impressions", "Clicks", "Conversions", "Revenue"]).copy()
df = df[(df["Campaign_Cost"] > 0) & (df["Revenue"] >= 0) & (df["Clicks"] > 0) & (df["Impressions"] > 0)]

# Metric calculations
df["CTR_%"] = (df["Clicks"] / df["Impressions"]) * 100.0
df["Conversion_Rate_%"] = (df["Conversions"] / df["Clicks"]) * 100.0
df["CPA"] = df["Campaign_Cost"] / df["Conversions"]
df["ROAS"] = df["Revenue"] / df["Campaign_Cost"]
df["Net_Profit"] = df["Revenue"] - df["Campaign_Cost"]
df["High_ROAS"] = (df["ROAS"] >= 2.5).astype(int)

print(f"Cleaned dataset: {len(df)} records")
df.head()

## 3. Campaign Performance Comparison

In [ ]:
cmp_summary = df.groupby("Campaign_ID").agg({
    "Campaign_Cost": "sum",
    "Impressions": "sum",
    "Clicks": "sum",
    "Conversions": "sum",
    "Revenue": "sum",
    "CTR_%": "mean",
    "Conversion_Rate_%": "mean",
    "CPA": "mean",
    "ROAS": "mean"
}).reset_index()

cmp_summary["Overall_ROAS"] = cmp_summary["Revenue"] / cmp_summary["Campaign_Cost"]
cmp_summary.sort_values(by="Overall_ROAS", ascending=False)

## 4. Visualizations: CTR vs Revenue Investigation & Impact Quadrants

In [ ]:
# Visualization: Does High CTR mean High Revenue?
plt.figure(figsize=(10, 6))
sns.scatterplot(data=df, x="CTR_%", y="Revenue", hue="Campaign_ID", size="Conversions", sizes=(50, 300), alpha=0.85)
plt.title("Investigation: CTR (%) vs Revenue ($)", fontsize=14, fontweight="bold")
plt.xlabel("Click-Through Rate (CTR %)")
plt.ylabel("Revenue ($)")
plt.show()

## 5. Machine Learning: Decision Tree Classifier

In [ ]:
features = ["Campaign_Cost", "Impressions", "Clicks", "Conversions", "CTR_%", "Conversion_Rate_%", "Customer_Segment", "Campaign_ID"]
X = pd.get_dummies(df[features], drop_first=True)
y = df["High_ROAS"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

dt = DecisionTreeClassifier(max_depth=4, criterion="gini", random_state=42)
dt.fit(X_train, y_train)

y_pred = dt.predict(X_test)
print(classification_report(y_test, y_pred))

# Feature Importance
imp = pd.Series(dt.feature_importances_, index=X.columns).sort_values(ascending=False)
print("Top Feature Importances:")
print(imp.head(6))

## 6. Decision Tree Visualization

In [ ]:
plt.figure(figsize=(16, 8))
plot_tree(dt, feature_names=list(X.columns), class_names=["Low/Mod ROAS", "High ROAS"], filled=True, rounded=True, fontsize=8)
plt.title("Trained Decision Tree for Campaign Success (High ROAS)", fontsize=14, fontweight="bold")
plt.show()